In [ ]:
import pandas as pd
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    df = pd.read_csv('data/corpus/master_corpus.csv')
    print(f"✅ Successfully loaded! Total records: {len(df)}.\n")
except FileNotFoundError:
    print("❌ Cannot find 'data/corpus/master_corpus.csv'. Please ensure the previous data cleaning script successfully generated this file!")
    raise

test_df = df[df['Keyword'] == 'Volk'].sample(3, random_state=42).copy()

def analyze_sentence(sentence, keyword, year):
    prompt = f"""
    Du bist ein Experte für die deutsche Begriffsgeschichte und politische Linguistik.
    Analysiere den folgenden Satz im Kontext der deutschen Nachkriegszeit (Jahr: {year}).
    Fokus-Wort: "{keyword}"
    
    Satz: "{sentence}"
    
    Bewerte das Fokus-Wort in diesem Satz nach folgenden DREI Kriterien:
    
    1. Nationalistische Intensität (intensity): Tendiert der Kontext eher zur "Blutsgemeinschaft" oder zur "Verfassungsgemeinschaft"? 
       Antworte mit einem ganzzahligen Wert von 1 bis 5. 
       (1 = rein staatsbürgerlich/demokratisch/Verfassungsgemeinschaft, 3 = alltäglich/Bevölkerung, 5 = stark ethnisch/biologisch/Blutsgemeinschaft).
       
    2. Emotionale Valenz (valence): Erscheint das Wort als "Symbol des Stolzes" oder als "Objekt der Reflexion/Mahnung"?
       Antworte mit einem ganzzahligen Wert von 1 bis 5.
       (1 = Objekt der Mahnung/Kritik/historisch belastet, 3 = neutral/deskriptiv, 5 = Symbol des Stolzes/identitätsstiftend).
       
    3. Temporalität (temporality): Wird das Konzept eher als etwas "Vergangenes" oder als etwas "Zukünftiges/Konstruktives" dargestellt?
       Wähle exakt eine dieser drei Kategorien: "historisch/überholt", "gegenwärtig/neutral", "zukünftig/konstruktiv".
       
    4. Begründung (reasoning): Ein kurzer, präziser Satz zur Begründung deiner Bewertung.
    
    Antworte AUSSCHLIESSLICH im strengen JSON-Format. Beispiel:
    {{"intensity": 1, "valence": 3, "temporality": "gegenwärtig/neutral", "reasoning": "Das Wort bezieht sich hier auf die Wähler als demokratischen Souverän, ohne ethnische Aufladung."}}
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",  
            messages=[
                {"role": "system", "content": "You are a helpful assistant that outputs strictly in JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={ "type": "json_object" } 
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'{{"error": "{str(e)}"}}'

print("🤖 LLM (GPT-5 Mini) starting to review test data...\n")
print("=" * 70)

for index, row in test_df.iterrows():
    print(f"Year: {row['Year']}")
    print(f"Original Sentence: {row['Hit_Cleaned']}")
    
    result_str = analyze_sentence(row['Hit_Cleaned'], row['Keyword'], row['Year'])
    
    try:
        result_json = json.loads(result_str)
        
        if "error" in result_json:
            print(f"API Error: {result_json['error']}")
        else:
            print(f"Nationalist Intensity (1-5): {result_json.get('intensity')}")
            print(f"Emotional Valence (1-5): {result_json.get('valence')}")
            print(f"Temporality: {result_json.get('temporality')}")
            print(f"Reasoning: {result_json.get('reasoning')}")
            
    except json.JSONDecodeError:
        print(f"Parsing failed. The model did not return standard JSON. Original response: {result_str}")
        
    print("=" * 70)
    
print("✅ Testing completed!")